In [2]:
from src.parquet_utils import *

In [7]:
row = {
    'instrument_token': 53490439,
    'mode': 'full',
    'volume': 12510,
    'last_price': 4084.0,
    'average_price': 4086.55,
    'last_quantity': 1,
    'buy_quantity': 2356,
    'sell_quantity': 2440,
    'change': 0.46740467404674046,
    'last_trade_time': datetime(2018, 1, 15, 13, 16, 54),
    'timestamp': datetime(2018, 1, 15, 13, 16, 56),
    'oi': 21845,
    'oi_day_low': 0,
    'oi_day_high': 0,
    'ohlc': {
        'high': 4093.0,
        'close': 4065.0,
        'open': 4088.0,
        'low': 4080.0
    },
    'tradable': True,
    'depth': {
        'sell': [{
            'price': 4085.0,
            'orders': 1048576,
            'quantity': 43
        }, {
            'price': 4086.0,
            'orders': 2752512,
            'quantity': 134
        }, {
            'price': 4087.0,
            'orders': 1703936,
            'quantity': 133
        }, {
            'price': 4088.0,
            'orders': 1376256,
            'quantity': 70
        }, {
            'price': 4089.0,
            'orders': 1048576,
            'quantity': 46
        }],
        'buy': [{
            'price': 4084.0,
            'orders': 589824,
            'quantity': 53
        }, {
            'price': 4083.0,
            'orders': 1245184,
            'quantity': 145
        }, {
            'price': 4082.0,
            'orders': 1114112,
            'quantity': 63
        }, {
            'price': 4081.0,
            'orders': 1835008,
            'quantity': 69
        }, {
            'price': 4080.0,
            'orders': 2752512,
            'quantity': 89
        }]
    }
}

In [8]:
T = pa.Table.from_pylist([row])

In [10]:
writer = StreamingParquetWriter("/tmp/test", T.schema)

In [11]:
for i in range(1_000_000):
    writer.write_row(row)

In [13]:
!ls /tmp/test/date\=2025-11-28

part-00000.parquet part-00002.parquet part-00004.parquet
part-00001.parquet part-00003.parquet


In [15]:
import pyarrow.parquet as pq
import os, glob

parts = sorted(glob.glob("/tmp/test/date=2025-11-28/part-*.parquet"))
total = sum(pq.ParquetFile(p).metadata.num_rows for p in parts)
print(len(parts), total)  # expect 5, 1000000

5 1000000


In [16]:
def load_day(path: str):  # path = "/data/ticks/date=2025-11-28"
    files = sorted(glob.glob(os.path.join(path, "part-*.parquet")))
    if not files:
        return pa.Table.from_arrays([], names=[])
    tables = [pq.read_table(f) for f in files]
    return pa.concat_tables(tables, promote=True)

In [17]:
tab = load_day("/tmp/test/date=2025-11-28/")

/var/folders/w6/_170l_817t1bzzqd2cgxp76h0000gq/T/ipykernel_53485/3216587386.py:1: FutureWarning: promote has been superseded by promote_options='default'.
  tab = load_day("/tmp/test/date=2025-11-28/")


In [27]:
tab.slice(0, 3)['last_price']

[
  [
    4084,
    4084,
    4084
  ]
]